In [1]:
import numpy as np
import pandas as pd
import tensorflow
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.interpolate import griddata
import unittest
import math
from scipy.stats import mvn

# 1. Import raw data

In [2]:
df = pd.read_csv(r"C:\Users\victo\Downloads\spy_2020_2022.csv", low_memory=False)

# 2. Clean data

In [3]:
df.columns = df.columns.str.strip().str.strip('[]')
df = df.drop(['QUOTE_UNIXTIME', 'QUOTE_READTIME', 'QUOTE_TIME_HOURS', 'EXPIRE_UNIX', 'C_DELTA', 'C_GAMMA', 'C_VEGA', 'C_THETA', 'C_RHO', 'P_DELTA', 'P_GAMMA', 'P_VEGA', 'P_THETA', 'P_RHO'], axis=1)
df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)
df['EXPIRE_DATE'] = pd.to_datetime(df['EXPIRE_DATE'], errors='coerce')
df['QUOTE_DATE'] = pd.to_datetime(df['QUOTE_DATE'], errors='coerce')
date_cols = ['EXPIRE_DATE', 'QUOTE_DATE']
cols_to_exclude = ['EXPIRE_DATE', 'QUOTE_DATE', 'C_SIZE', 'P_SIZE']
num_cols = df.columns.difference(cols_to_exclude)
df[date_cols] = df[date_cols].apply(pd.to_datetime, format='%Y-%m-%d', errors='coerce')
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df = df.drop(['C_IV', 'C_VOLUME', 'C_LAST', 'C_SIZE', 'C_BID', 'C_ASK', 'STRIKE_DISTANCE', 'STRIKE_DISTANCE_PCT', 'EXPIRE_DATE'], axis=1)
df['moneyness'] = df['UNDERLYING_LAST'] / df['STRIKE']
df = df[df['P_BID'] > 0]  # filter out 0 bid options
df['midprice'] = (df['P_BID'] + df['P_ASK']) / 2.0
df = df[df['midprice'] > 1] 
df = df[df['DTE'] > 0]  # filter out 0DTE options
df = df[df['P_VOLUME'] > 0] # filter out 0 volume options
df = df[df['P_IV'] < 3] # filter out extreme IV values
df = df[df['P_IV'] > 0] # filter out extreme IV values
df['T'] = df['DTE'] / 365.0

C:\Users\victo\AppData\Local\Temp\ipykernel_25728\2765435807.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lstrip() if isinstance(x, str) else x)


# 3. Add additional data and clean

In [4]:
df['log_moneyness'] = np.log(df['moneyness'])
df['q'] = 0.015  # constant dividend yield
rf = pd.read_csv(r"C:\Users\victo\Downloads\DTB3.csv", low_memory=False)
rf['observation_date'] = pd.to_datetime(rf['observation_date'], format='%Y-%m-%d', errors='coerce')
rf.rename(columns={'observation_date': 'date', 'DTB3': 'risk_free_rate'}, inplace=True)
rf['risk_free_rate'] /= 100
rf.sort_values('date', inplace=True)
rf.fillna(method='ffill', inplace=True)
df.rename(columns={'QUOTE_DATE': 'date'}, inplace=True)
df = df.merge(rf, on='date', how='left')

C:\Users\victo\AppData\Local\Temp\ipykernel_25728\3430148834.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  rf.fillna(method='ffill', inplace=True)


# 4. Define Black-Scholes implied volatility

In [5]:
from scipy.optimize import brentq, newton
def black_scholes_put_price(S, K, T, r, sigma):
    """Black-Scholes formula for European put option."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

def implied_volatility_put(market_price, S, K, T, r, 
                           initial_guess=0.2, 
                           tol=1e-6, 
                           max_iter=100):
    """Compute implied volatility using the Newton-Raphson method, with fallback."""
    
    def objective(sigma):
        return black_scholes_put_price(S, K, T, r, sigma) - market_price

    try:
        # Try Newton-Raphson with derivative
        iv = newton(objective, initial_guess, tol=tol, maxiter=max_iter)
    except (RuntimeError, OverflowError):
        # Fallback to Brent's method if Newton-Raphson fails
        iv = brentq(objective, 1e-6, 5.0, xtol=tol)
    return iv

# 5. Get correct BS implied vol from market prices

In [6]:
def compute_iv(row):
    try:
        iv = implied_volatility_put(row['midprice'], row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], initial_guess=0.2, tol=1e-3, max_iter=100)
    except Exception as e:
        iv = np.nan
    return iv
df['implied_volatility'] = df.apply(compute_iv, axis=1)

# 6. Split data

In [7]:
# Sort by date
df = df.sort_values('date')

# Determine cutoff date (e.g., 80% point)
cutoff_date = df['date'].quantile(0.8)

# Create splits
train_df = df[df['date'] <= cutoff_date]
test_df = df[df['date'] > cutoff_date]

# 7. Define Bjerklund-Stensland model

In [8]:
def _phi(fs, t, gamma, h, i, r, b, v):
    d1 = -(math.log(fs / h) + (b + (gamma - 0.5) * (v ** 2)) * t) / (v * math.sqrt(t))
    d2 = d1 - 2 * math.log(i / fs) / (v * math.sqrt(t))

    lambda1 = (-r + gamma * b + 0.5 * gamma * (gamma - 1) * (v ** 2))
    kappa = (2 * b) / (v ** 2) + (2 * gamma - 1)

    phi = math.exp(lambda1 * t) * (fs ** gamma) * (norm.cdf(d1) - ((i / fs) ** kappa) * norm.cdf(d2))
    return phi

def _cbnd(a, b, rho):
    # This distribution uses the Genz multi-variate normal distribution 
    # code found as part of the standard SciPy distribution
    lower = np.array([0, 0])
    upper = np.array([a, b])
    infin = np.array([0, 0])
    correl = rho
    error, value, inform = mvn.mvndst(lower, upper, infin, correl)
    return value


def _psi(fs, t2, gamma, h, i2, i1, t1, r, b, v):
    vsqrt_t1 = v * math.sqrt(t1)
    vsqrt_t2 = v * math.sqrt(t2)

    bgamma_t1 = (b + (gamma - 0.5) * (v ** 2)) * t1
    bgamma_t2 = (b + (gamma - 0.5) * (v ** 2)) * t2

    d1 = (math.log(fs / i1) + bgamma_t1) / vsqrt_t1
    d3 = (math.log(fs / i1) - bgamma_t1) / vsqrt_t1

    d2 = (math.log((i2 ** 2) / (fs * i1)) + bgamma_t1) / vsqrt_t1
    d4 = (math.log((i2 ** 2) / (fs * i1)) - bgamma_t1) / vsqrt_t1

    e1 = (math.log(fs / h) + bgamma_t2) / vsqrt_t2
    e2 = (math.log((i2 ** 2) / (fs * h)) + bgamma_t2) / vsqrt_t2
    e3 = (math.log((i1 ** 2) / (fs * h)) + bgamma_t2) / vsqrt_t2
    e4 = (math.log((fs * (i1 ** 2)) / (h * (i2 ** 2))) + bgamma_t2) / vsqrt_t2

    tau = math.sqrt(t1 / t2)
    lambda1 = (-r + gamma * b + 0.5 * gamma * (gamma - 1) * (v ** 2))
    kappa = (2 * b) / (v ** 2) + (2 * gamma - 1)

    psi = math.exp(lambda1 * t2) * (fs ** gamma) * (_cbnd(-d1, -e1, tau)
                                                    - ((i2 / fs) ** kappa) * _cbnd(-d2, -e2, tau)
                                                    - ((i1 / fs) ** kappa) * _cbnd(-d3, -e3, -tau)
                                                    + ((i1 / i2) ** kappa) * _cbnd(-d4, -e4, -tau))
    return psi


def _bjerksund_stensland_2002(fs, x, t, r, b, v):
    # preliminary calculations
    v2 = v ** 2
    t1 = 0.5 * (math.sqrt(5) - 1) * t
    t2 = t

    beta_inside = ((b / v2 - 0.5) ** 2) + 2 * r / v2
    # forcing the inside of the sqrt to be a positive number
    beta_inside = abs(beta_inside)
    beta = (0.5 - b / v2) + math.sqrt(beta_inside)
    b_infinity = (beta / (beta - 1)) * x
    b_zero = max(x, (r / (r - b)) * x)

    h1 = -(b * t1 + 2 * v * math.sqrt(t1)) * ((x ** 2) / ((b_infinity - b_zero) * b_zero))
    h2 = -(b * t2 + 2 * v * math.sqrt(t2)) * ((x ** 2) / ((b_infinity - b_zero) * b_zero))

    i1 = b_zero + (b_infinity - b_zero) * (1 - math.exp(h1))
    i2 = b_zero + (b_infinity - b_zero) * (1 - math.exp(h2))

    alpha1 = (i1 - x) * (i1 ** (-beta))
    alpha2 = (i2 - x) * (i2 ** (-beta))

    # check for immediate exercise
    if fs >= i2:
        value = fs - x
    else:
        # Perform the main calculation    
        value = (alpha2 * (fs ** beta)
                 - alpha2 * _phi(fs, t1, beta, i2, i2, r, b, v)
                 + _phi(fs, t1, 1, i2, i2, r, b, v)
                 - _phi(fs, t1, 1, i1, i2, r, b, v)
                 - x * _phi(fs, t1, 0, i2, i2, r, b, v)
                 + x * _phi(fs, t1, 0, i1, i2, r, b, v)
                 + alpha1 * _phi(fs, t1, beta, i1, i2, r, b, v)
                 - alpha1 * _psi(fs, t2, beta, i1, i2, i1, t1, r, b, v)
                 + _psi(fs, t2, 1, i1, i2, i1, t1, r, b, v)
                 - _psi(fs, t2, 1, x, i2, i1, t1, r, b, v)
                 - x * _psi(fs, t2, 0, i1, i2, i1, t1, r, b, v)
                 + x * _psi(fs, t2, 0, x, i2, i1, t1, r, b, v))

        
    return value

def BS2002(type, fs, x, t, r, b, v):
    if type == 'c':
        return _bjerksund_stensland_2002(fs, x, t, r, b, v)
    elif type == 'p':
        put__x = fs
        put_fs = x
        put_b = -b
        put_r = r - b
        return _bjerksund_stensland_2002(put_fs, put__x, t, put_r, put_b, v)

# 8. Apply Bjerksund-Stensland model

In [9]:
test_df['Bjerksund_Stensland_price'] = test_df.apply(lambda row: BS2002('p', row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, 0.2), axis=1)

C:\Users\victo\AppData\Local\Temp\ipykernel_25728\2514774326.py:18: DeprecationWarning: `scipy.stats.mvn.mvndst` is deprecated along with the `scipy.stats.mvn` namespace. `scipy.stats.mvn.mvndst` will be removed in SciPy 1.14.0, and the `scipy.stats.mvn` namespace will be removed in SciPy 2.0.0.
  error, value, inform = mvn.mvndst(lower, upper, infin, correl)
C:\Users\victo\AppData\Local\Temp\ipykernel_25728\1191338202.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['Bjerksund_Stensland_price'] = test_df.apply(lambda row: BS2002('p', row['UNDERLYING_LAST'], row['STRIKE'], row['T'], row['risk_free_rate'], 0.015, 0.2), axis=1)


# 9. Calculate error metrics

In [10]:
df = df.dropna()
test_df = test_df.dropna()
train_df = train_df.dropna()

In [ ]:
# Errors
abs_error_sum = 0
rmse_sum = 0
rel_error_sum = 0
for i in range(len(test_df)):
    abs_error_sum += np.abs(test_df['implied_volatility'].iloc[i] - 0.2)
    rmse_sum += (test_df['implied_volatility'].iloc[i] - 0.2)**2
    rel_error_sum += np.abs((test_df['implied_volatility'].iloc[i] - 0.2) / np.abs(test_df['implied_volatility'].iloc[i]))

# Metrics
mae = abs_error_sum / len(test_df)
rmse = np.sqrt(rmse_sum / len(test_df))
rel_error = rel_error_sum / len(test_df)
re_sum = rel_error_sum


print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Sum of RE: {re_sum:.4f}")
print(f"Mean RE: {rel_error:.4f}")

MAE: 0.0873
RMSE: 0.1260
Sum of RE: 80997.9723
Mean RE: 0.2727


: 

In [12]:
test_df.head()

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,moneyness,midprice,T,log_moneyness,q,risk_free_rate,implied_volatility,Bjerksund_Stensland_price
934115,2022-02-17,437.17,132.96,420.0,18.32,18.44,50 x 23,17.81,0.24786,12.0,1.040881,18.380,0.364274,0.040067,0.015,0.0037,0.255325,12.376310
934114,2022-02-17,437.17,132.96,418.0,17.74,17.87,45 x 50,16.17,0.25105,1.0,1.045861,17.805,0.364274,0.044841,0.015,0.0037,0.257968,11.637081
934113,2022-02-17,437.17,132.96,416.0,17.03,18.06,5 x 42,15.30,0.25678,27.0,1.050889,17.545,0.364274,0.049637,0.015,0.0037,0.263684,10.927794
934106,2022-02-17,437.17,132.96,399.0,13.03,13.14,84 x 56,12.99,0.27647,42.0,1.095664,13.085,0.364274,0.091361,0.015,0.0037,0.282159,6.051598
934109,2022-02-17,437.17,132.96,405.0,14.40,14.48,27 x 27,13.82,0.26817,4.0,1.079432,14.440,0.364274,0.076435,0.015,0.0037,0.274699,7.545548


# 10. Define Neural Network

In [34]:
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader


class OptionpricingDataset(Dataset):
    def __init__(self, df, scaler=None, fit_scaler=False):
        df = df.copy()
        self.scaler = scaler
        self.features_cols = ['log_moneyness', 'T', 'risk_free_rate', 'q', 'UNDERLYING_LAST', 'STRIKE', 'P_VOLUME']
        self.target_col = ['implied_volatility']
        X = df[self.features_cols].values.astype(np.float32)
        y = df[self.target_col].values.astype(np.float32).reshape(-1, 1)

        # apply scaling
        if scaler is None:
            self.scaler = StandardScaler()
            self.X = self.scaler.fit_transform(X)
        else:
            self.scaler = scaler
            if fit_scaler:
                self.X = self.scaler.fit_transform(X)
            else:
                self.X = self.scaler.transform(X)

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = OptionpricingDataset(train_df, fit_scaler=True)
scaler = train_dataset.scaler  # Save the fitted scaler

test_dataset = OptionpricingDataset(test_df, scaler=scaler, fit_scaler=False)

import torch.nn as nn

class OptionPricingNN(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[40,40,40,40,]):
        super(OptionPricingNN, self).__init__()
        layers = []
        in_dim = input_dim

        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 1))  # Output layer
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def train_model(model, train_dataset, test_dataset, epochs=100, batch_size=10000, lr=1e-3):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    def mape_loss(y_pred, y_true, eps=1e-6):
        return torch.mean(torch.abs((y_true - y_pred) / (y_true + eps)))
    class RMSELoss(nn.Module):
        def __init__(self, eps=1e-6):
            super(RMSELoss, self).__init__()
            self.mse = nn.MSELoss()
            self.eps = eps

        def forward(self, y_pred, y_true):
            return torch.sqrt(self.mse(y_pred, y_true) + self.eps)
     #criterion = nn.SmoothL1Loss()  # Use SmoothL1Loss for regression
    criterion = RMSELoss()  # Use RMSE loss for regression

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        avg_train_loss = train_loss / len(train_dataset)

        # Evaluation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)

        avg_test_loss = test_loss / len(test_dataset)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.3f} | Test Loss: {avg_test_loss:.3f}")

    return model

In [41]:
input_dim = len(train_dataset.features_cols)
model = OptionPricingNN(input_dim=input_dim)

# Train model
trained_model = train_model(model, train_dataset, test_dataset, epochs=100)

Epoch 1/100 | Train Loss: 0.108 | Test Loss: 0.137
Epoch 2/100 | Train Loss: 0.061 | Test Loss: 0.109
Epoch 3/100 | Train Loss: 0.051 | Test Loss: 0.118
Epoch 4/100 | Train Loss: 0.048 | Test Loss: 0.122
Epoch 5/100 | Train Loss: 0.045 | Test Loss: 0.141
Epoch 6/100 | Train Loss: 0.043 | Test Loss: 0.146
Epoch 7/100 | Train Loss: 0.041 | Test Loss: 0.144
Epoch 8/100 | Train Loss: 0.040 | Test Loss: 0.161
Epoch 9/100 | Train Loss: 0.038 | Test Loss: 0.149
Epoch 10/100 | Train Loss: 0.038 | Test Loss: 0.145
Epoch 11/100 | Train Loss: 0.037 | Test Loss: 0.154
Epoch 12/100 | Train Loss: 0.036 | Test Loss: 0.135
Epoch 13/100 | Train Loss: 0.036 | Test Loss: 0.146
Epoch 14/100 | Train Loss: 0.035 | Test Loss: 0.129
Epoch 15/100 | Train Loss: 0.051 | Test Loss: 0.140
Epoch 16/100 | Train Loss: 0.042 | Test Loss: 0.123
Epoch 17/100 | Train Loss: 0.040 | Test Loss: 0.121
Epoch 18/100 | Train Loss: 0.038 | Test Loss: 0.126
Epoch 19/100 | Train Loss: 0.037 | Test Loss: 0.126
Epoch 20/100 | Train 

In [40]:
len(test_df)

296973

In [42]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def get_prediction_table(model, test_dataset, original_df=None, num_rows=10000000):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    X = test_dataset.X.to(device)
    y_true = test_dataset.y.cpu().numpy()
    
    with torch.no_grad():
        y_pred = model(X).cpu().numpy()

    # Compute errors
    abs_error = np.abs(y_true.flatten() - y_pred.flatten())
    rel_error = abs_error / np.maximum(np.abs(y_true.flatten()), 1e-6)

    mae = np.mean(abs_error)
    rmse = np.sqrt(np.mean((y_true.flatten() - y_pred.flatten()) ** 2))
    rel_error_sum = np.sum(rel_error)

    print(f"\nMAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"Sum of Relative Errors: {rel_error_sum:.6f}")

    # Create table
    data = {
        'Actual Price': y_true.flatten(),
        'Predicted Price': y_pred.flatten(),
        'Absolute Error': abs_error,
        'Relative Error': rel_error
    }

    df_result = pd.DataFrame(data)

    # Optionally join original metadata
    if original_df is not None:
        original_df = original_df.reset_index(drop=True)
        df_result = pd.concat([original_df.reset_index(drop=True), df_result], axis=1)

    return df_result.head(num_rows)

prediction_table = get_prediction_table(trained_model, test_dataset, original_df=test_df)




MAE: 0.297686
RMSE: 0.513688
Sum of Relative Errors: 355720.781250


In [ ]:
prediction_table.sort_values(by=['Relative Error'], ascending=False, inplace=True)

In [32]:
prediction_table.sort_values(by=['Absolute Error'], ascending=False, inplace=True)
prediction_table.head(n=10)

,date,UNDERLYING_LAST,DTE,STRIKE,P_BID,P_ASK,P_SIZE,P_LAST,P_IV,P_VOLUME,...,T,log_moneyness,q,risk_free_rate,implied_volatility,Bjerksund_Stensland_price,Actual Price,Predicted Price,Absolute Error,Relative Error
284867,2022-12-15,389.71,1.0,605.0,217.10,217.39,8 x 8,211.98,0.00162,8.0,...,0.002740,-0.439826,0.015,0.0422,4.829312,215.29,4.829312,0.509079,4.320232,0.894585
284868,2022-12-15,389.71,1.0,590.0,201.85,202.13,8 x 12,196.99,0.00111,8.0,...,0.002740,-0.414720,0.015,0.0422,4.498366,200.29,4.498366,0.485075,4.013291,0.892166
34709,2022-03-17,440.76,1.0,650.0,209.74,211.86,42 x 42,212.58,0.00113,17.0,...,0.002740,-0.388472,0.015,0.0040,4.088702,209.24,4.088702,0.961067,3.127635,0.764946
116210,2022-06-15,379.15,2.0,685.0,306.80,308.05,1 x 1,308.07,0.00072,1.0,...,0.005479,-0.591487,0.015,0.0169,4.163850,305.85,4.163850,1.079603,3.084246,0.740720
116165,2022-06-15,379.15,2.0,680.0,301.80,303.05,1 x 1,303.05,0.00110,1.0,...,0.005479,-0.584161,0.015,0.0169,4.123011,300.85,4.123011,1.058996,3.064014,0.743150
117765,2022-06-16,366.89,1.0,500.0,133.36,135.46,10 x 1,134.22,0.00110,6.0,...,0.002740,-0.309546,0.015,0.0154,3.429547,133.11,3.429547,0.504025,2.925522,0.853034
34710,2022-03-17,440.76,1.0,630.0,189.74,191.86,42 x 42,192.45,0.00181,5.0,...,0.002740,-0.357219,0.015,0.0040,3.827632,189.24,3.827632,0.932417,2.895215,0.756398
117766,2022-06-16,366.89,1.0,480.0,114.28,115.47,5 x 1,115.36,1.92321,4.0,...,0.002740,-0.268724,0.015,0.0154,3.269190,113.11,3.269190,0.403720,2.865469,0.876508
284869,2022-12-15,389.71,1.0,500.0,111.90,112.14,8 x 8,108.32,0.00181,1.0,...,0.002740,-0.249205,0.015,0.0422,3.048103,110.29,3.048103,0.379040,2.669063,0.875647
117770,2022-06-16,366.89,1.0,465.0,99.28,100.47,5 x 1,102.19,1.73509,3.0,...,0.002740,-0.236975,0.015,0.0154,2.972896,98.11,2.972896,0.346022,2.626874,0.883608
